# ConvNeXt-Tiny v4 — ConvNeXt 장점 극대화

## v3 → v4 변경 사항

| 항목 | v3 (82.91%) | v4 (이번) | 근거 |
|------|------------|-----------|------|
| LR 전략 | Backbone 균일 LR | **LLRD** (Layer-wise LR Decay) | ConvNeXt 공식 학습법 — 앞 레이어 보존, 뒤 레이어 집중 업데이트 |
| 증강 | 4개 | **+ RandomErasing** | GradCAM 분석 결과: 모델이 로고·버튼에 집중 → 실루엣 집중 유도 |
| GradCAM | 자체 구현 | **팀원 gradcam.py 방식** | output[0,idx].backward() 직접 호출, overlay_cam 통일 |


## 0. 공통 설정

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tqdm import tqdm
from PIL import Image, ImageOps
from collections import defaultdict, Counter

# ── 재현성
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── 경로
DATA_DIR  = './preprocess'
TEST_DIR  = './test_final'
SAVE_DIR  = './checkpoints_v4'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 하이퍼파라미터
IMG_SIZE    = 224
BATCH_SIZE  = 32
WARMUP_EP   = 5
FINETUNE_EP = 40
LR_HEAD     = 1e-3
LR_FULL     = 5e-5
LLRD_DECAY  = 0.8      # ⭐ v4 추가: 레이어별 LR 감쇠율
VAL_RATIO   = 0.2
NUM_WORKERS = 0
PATIENCE    = 10

CLASSES     = ['mug', 'straight', 'taper_smooth', 'taper_step']
NUM_CLASSES = len(CLASSES)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

print('설정 완료')


## 1. Transform 정의

> **v4 추가**: `RandomErasing` — GradCAM 분석 결과 모델이 로고·버튼·빨대에 집중하는 문제 해소
> 나머지 4개 증강은 v3와 동일 유지 (RandomPerspective 핵심 보존)


In [ ]:
NORMALIZE = transforms.Normalize(mean=MEAN, std=STD)

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomPerspective(distortion_scale=0.35, p=0.4),  # ⭐ v3 핵심 유지
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    NORMALIZE,
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),           # ⭐ v4 추가: 부분 가림
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    NORMALIZE,
])

print('Transform 정의 완료')
print('train: RandomResizedCrop → RandomHorizontalFlip → RandomPerspective → ColorJitter → RandomErasing')


## 2. 데이터로더 (Stratified Split + EXIF 보정)

In [ ]:
def exif_safe_loader(path):
    """EXIF 방향 보정 (PIL 기본값은 EXIF를 무시하므로 명시적으로 처리)"""
    with Image.open(path) as img:
        img = ImageOps.exif_transpose(img)
        return img.convert('RGB')


class ConnectorDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(exif_safe_loader(path)), label


def list_samples(root):
    samples = []
    for cls in CLASSES:
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir):
            continue
        for name in sorted(os.listdir(cls_dir)):
            if os.path.splitext(name)[1].lower() in ('.jpg', '.jpeg', '.png'):
                samples.append((os.path.join(cls_dir, name), CLASSES.index(cls)))
    return samples


# Stratified Split
all_samples = list_samples(DATA_DIR)
by_class    = defaultdict(list)
for s in all_samples:
    by_class[s[1]].append(s)

rng = random.Random(SEED)
train_samples, val_samples = [], []
for items in by_class.values():
    items = items[:]
    rng.shuffle(items)
    n_val = max(1, int(len(items) * VAL_RATIO))
    val_samples.extend(items[:n_val])
    train_samples.extend(items[n_val:])

train_loader = DataLoader(ConnectorDataset(train_samples, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(ConnectorDataset(val_samples, eval_tf),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_samples)}장  |  Val: {len(val_samples)}장')
tr_cnt = Counter([s[1] for s in train_samples])
for i, cls in enumerate(CLASSES):
    print(f'  {cls:14s}: train {tr_cnt[i]}장')


## 3. 모델 정의 (ConvNeXt-Tiny)

In [ ]:
torch.manual_seed(SEED)
model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

total = sum(p.numel() for p in model.parameters())
print(f'ConvNeXt-Tiny | 총 파라미터: {total/1e6:.1f}M')

# ConvNeXt features 구조 확인 (LLRD용)
print('\nfeatures 스테이지 구조:')
for i, stage in enumerate(model.features):
    params = sum(p.numel() for p in stage.parameters()) / 1e6
    print(f'  features[{i}]: {type(stage).__name__}  ({params:.2f}M params)')


## 4. LLRD 파라미터 그룹 설정

> **Layer-wise LR Decay** — ConvNeXt 공식 학습 기법
> 앞 레이어(기본 엣지/텍스처)는 LR을 낮게 → 사전학습 지식 보존
> 뒤 레이어(형태 판단)는 LR을 높게 → 커넥터 형태에 집중 업데이트


In [ ]:
def build_llrd_params(model, base_lr, decay=LLRD_DECAY):
    """ConvNeXt features[0..7] 에 레이어별 감쇠 LR 적용.
    앞 레이어일수록 base_lr * decay^(n-i) 로 낮아짐.
    classifier 는 base_lr 그대로.
    """
    param_groups = []
    stages = list(model.features.children())   # features[0] ~ features[7]
    n = len(stages)
    for i, stage in enumerate(stages):
        lr_i = base_lr * (decay ** (n - 1 - i))
        param_groups.append({'params': list(stage.parameters()), 'lr': lr_i})

    param_groups.append({'params': list(model.classifier.parameters()), 'lr': base_lr})
    return param_groups


# LR 분포 미리 확인
print(f'LLRD decay={LLRD_DECAY}, base_lr={LR_FULL:.2e}')
print('레이어별 LR:')
stages = list(model.features.children())
n = len(stages)
for i, stage in enumerate(stages):
    lr_i = LR_FULL * (LLRD_DECAY ** (n - 1 - i))
    print(f'  features[{i}]: {lr_i:.2e}')
print(f'  classifier:  {LR_FULL:.2e}')


## 5. 학습 함수

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            out  = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, labels)
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


print('학습 함수 정의 완료')


## 6. Stage 1: Warmup (Head만 학습)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

for name, p in model.named_parameters():
    p.requires_grad = ('classifier' in name)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=1e-4
)
scaler    = GradScaler()
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=WARMUP_EP, eta_min=1e-6)

hist = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
best_acc, best_state, no_improve = 0.0, None, 0

print('=== Stage 1: Warmup (Head only) ===')
print(f"{'Epoch':>5}  {'Tr Loss':>8}  {'Tr Acc':>7}  {'Va Loss':>8}  {'Va Acc':>7}")
print('-' * 47)

for epoch in range(1, WARMUP_EP + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    scheduler.step()
    hist['train_loss'].append(tr_loss); hist['train_acc'].append(tr_acc)
    hist['val_loss'].append(va_loss);   hist['val_acc'].append(va_acc)
    if va_acc > best_acc:
        best_acc   = va_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    print(f"{epoch:>5}  {tr_loss:>8.4f}  {tr_acc*100:>6.2f}%  {va_loss:>8.4f}  {va_acc*100:>6.2f}%")

print(f'\nWarmup 완료 | Best Val Acc: {best_acc*100:.2f}%')


## 7. Stage 2: Finetune (LLRD 적용)

> v3: Backbone 전체에 동일 LR (`LR_FULL=5e-5`)
> v4: **LLRD** — `features[0]`은 `5e-5 × 0.8⁷ ≈ 2.1e-6`, `features[7]`은 `5e-5` 로 차등 적용


In [ ]:
for p in model.parameters():
    p.requires_grad = True

# ⭐ v4: LLRD 적용한 optimizer
optimizer = optim.AdamW(
    build_llrd_params(model, base_lr=LR_FULL * 10, decay=LLRD_DECAY),
    weight_decay=1e-4
)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EP, eta_min=1e-7)
scaler     = GradScaler()
no_improve = 0

print('=== Stage 2: Finetune (LLRD) ===')
print(f"{'Epoch':>5}  {'Tr Loss':>8}  {'Tr Acc':>7}  {'Va Loss':>8}  {'Va Acc':>7}")
print('-' * 47)

for epoch in tqdm(range(1, FINETUNE_EP + 1), desc='Finetune'):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    scheduler.step()
    hist['train_loss'].append(tr_loss); hist['train_acc'].append(tr_acc)
    hist['val_loss'].append(va_loss);   hist['val_acc'].append(va_acc)

    if va_acc > best_acc:
        best_acc   = va_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'\nEarly stopping @ epoch {epoch + WARMUP_EP}')
            break

    if epoch % 5 == 0:
        print(f"{epoch+WARMUP_EP:>5}  {tr_loss:>8.4f}  {tr_acc*100:>6.2f}%  {va_loss:>8.4f}  {va_acc*100:>6.2f}%")

save_path = os.path.join(SAVE_DIR, 'best_convnext_v4.pth')
torch.save(best_state, save_path)
print(f'\n최고 Val Acc: {best_acc*100:.2f}%  →  {save_path}')


## 8. 학습 곡선

In [ ]:
epochs = range(1, len(hist['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ConvNeXt-Tiny v4 학습 곡선', fontsize=13, fontweight='bold')

axes[0].plot(epochs, hist['train_loss'], label='Train', color='#3498DB')
axes[0].plot(epochs, hist['val_loss'],   label='Val',   color='#E74C3C')
axes[0].axvline(WARMUP_EP, color='gray', linestyle='--', alpha=0.5, label='Warmup 끝')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [v*100 for v in hist['train_acc']], label='Train', color='#3498DB')
axes[1].plot(epochs, [v*100 for v in hist['val_acc']],   label='Val',   color='#E74C3C')
axes[1].axvline(WARMUP_EP, color='gray', linestyle='--', alpha=0.5, label='Warmup 끝')
axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'train_curve_v4.png'), dpi=150)
plt.show()


## 9. Test 평가 — 클래스별 성능 분석

In [ ]:
model.load_state_dict(best_state)
model.eval()

test_samples   = list_samples(TEST_DIR)
test_img_paths = [s[0] for s in test_samples]
test_labels    = [s[1] for s in test_samples]

print(f'테스트 이미지 수: {len(test_samples)}장')
cnt = Counter(test_labels)
for i, cls in enumerate(CLASSES):
    print(f'  {cls:14s}: {cnt[i]}장')


@torch.no_grad()
def predict_single(img_path, model, transform):
    img   = exif_safe_loader(img_path)
    inp   = transform(img).unsqueeze(0).to(DEVICE)
    out   = model(inp)
    probs = torch.softmax(out, dim=1).squeeze().cpu().numpy()
    return int(probs.argmax()), probs


all_preds, all_probs = [], []
for path in tqdm(test_img_paths, desc='추론'):
    pred, probs = predict_single(path, model, eval_tf)
    all_preds.append(pred)
    all_probs.append(probs)

acc = sum(p == t for p, t in zip(all_preds, test_labels)) / len(test_labels)
print(f'\n전체 Test Accuracy: {acc*100:.2f}%')
print('\n=== Classification Report ===')
print(classification_report(test_labels, all_preds, target_names=CLASSES))

print('\n' + '='*50)
print('  클래스별 상세 결과')
print('='*50)
for cls_idx, cls_name in enumerate(CLASSES):
    cls_indices = [i for i, l in enumerate(test_labels) if l == cls_idx]
    correct = sum(all_preds[i] == cls_idx for i in cls_indices)
    total   = len(cls_indices)
    recall  = correct / total if total > 0 else 0
    status  = 'PASS ✅' if recall >= 0.7 else 'FAIL ❌'
    errors  = {}
    for i in cls_indices:
        if all_preds[i] != cls_idx:
            errors[CLASSES[all_preds[i]]] = errors.get(CLASSES[all_preds[i]], 0) + 1
    err_str = ', '.join(f'{k}:{v}' for k, v in errors.items()) if errors else '없음'
    print(f'  {status}  {cls_name:14s}  Recall {recall*100:5.1f}%  ({correct}/{total})')
    print(f'         오분류 → {err_str}')
print('='*50)


## 10. 클래스별 샘플 시각화 — 오분류 우선

In [ ]:
N_SHOW = 5
fig, axes = plt.subplots(NUM_CLASSES, N_SHOW, figsize=(3*N_SHOW, 4*NUM_CLASSES))
fig.suptitle('클래스별 예측 결과  (빨강=오분류  녹색=정답)', fontsize=14, fontweight='bold')

for cls_idx, cls_name in enumerate(CLASSES):
    cls_indices = [i for i, l in enumerate(test_labels) if l == cls_idx]
    wrong_idx   = [i for i in cls_indices if all_preds[i] != cls_idx]
    right_idx   = [i for i in cls_indices if all_preds[i] == cls_idx]
    show_idx    = (wrong_idx + right_idx)[:N_SHOW]
    cls_recall  = sum(all_preds[i] == cls_idx for i in cls_indices) / max(len(cls_indices), 1)

    for col, idx in enumerate(show_idx):
        img     = exif_safe_loader(test_img_paths[idx])
        correct = (all_preds[idx] == cls_idx)
        conf    = all_probs[idx][all_preds[idx]] * 100
        axes[cls_idx][col].imshow(img)
        axes[cls_idx][col].set_title(f'예측: {CLASSES[all_preds[idx]]}\n({conf:.1f}%)',
                                      fontsize=8, color='green' if correct else 'red')
        axes[cls_idx][col].axis('off')
        if col == 0:
            axes[cls_idx][col].set_ylabel(f'{cls_name}\nRecall {cls_recall*100:.0f}%',
                                           fontsize=9, rotation=90, va='center')
    for col in range(len(show_idx), N_SHOW):
        axes[cls_idx][col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'samples_v4.png'), dpi=120, bbox_inches='tight')
plt.show()


## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(test_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_title('Confusion Matrix — ConvNeXt v4', fontsize=12, fontweight='bold')
ax.set_ylabel('실제'); ax.set_xlabel('예측')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'cm_v4.png'), dpi=150)
plt.show()

sm_idx = CLASSES.index('taper_smooth')
st_idx = CLASSES.index('straight')
print(f'taper_smooth → straight 오분류: {cm[sm_idx][st_idx]}건')
print(f'straight → taper_smooth 오분류: {cm[st_idx][sm_idx]}건')


## 12. GradCAM — 팀원 gradcam.py 방식 적용

> **팀원 방식 그대로 적용**:
> - `output[0, target_idx].backward()` 직접 호출
> - `overlay_cam()` 함수: matplotlib jet 컬러맵으로 히트맵 생성
> - **target_layer**: `model.features[-1][-1]` (ConvNeXt 마지막 블록 = ResNet의 `layer4[-1]` 대응)
> - 클래스별 격자 — 맞은 것(O)과 틀린 것(X) 섞어서 표시


In [ ]:
# ── GradCAM 엔진 (팀원 gradcam.py 구조와 동일)
class GradCAM:
    """target_layer의 활성화·그래디언트로 클래스별 히트맵을 만든다."""

    def __init__(self, model, target_layer):
        self.model       = model
        self.activations = None
        self.gradients   = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inputs, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        output    = self.model(input_tensor)
        pred_idx  = int(output.argmax(dim=1).item())
        target_idx = pred_idx if class_idx is None else class_idx
        output[0, target_idx].backward()          # ← 팀원 방식 그대로

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = torch.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam     = cam[0, 0].cpu().numpy()
        cam    -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()
        return cam, pred_idx, output.softmax(dim=1)[0].detach().cpu().numpy()


def overlay_cam(rgb_uint8: np.ndarray, cam: np.ndarray) -> np.ndarray:
    """팀원 overlay_cam과 동일: matplotlib jet 컬러맵, 비율 0.45/0.55"""
    cam_resized = cv2.resize(cam, (rgb_uint8.shape[1], rgb_uint8.shape[0]))
    heatmap     = (plt.get_cmap('jet')(cam_resized)[:, :, :3] * 255).astype(np.uint8)
    return (0.45 * heatmap + 0.55 * rgb_uint8).astype(np.uint8)


# ── ConvNeXt target_layer: features[-1][-1] = ResNet의 layer4[-1] 대응
target_layer = model.features[-1][-1]
cam_engine   = GradCAM(model, target_layer)
print(f'GradCAM target: model.features[-1][-1] = {type(target_layer).__name__}')

# ── 클래스별 샘플 격자 (팀원 방식: 맞은 것 + 틀린 것 섞어서)
PER_CLASS = 4
target_cls = CLASSES   # 전체 4클래스

fig, axes = plt.subplots(len(target_cls), PER_CLASS,
                          figsize=(3 * PER_CLASS, 3 * len(target_cls)))
fig.suptitle('GradCAM v4 — 클래스별 판정 근거', fontsize=13, fontweight='bold')

# 클래스별 인덱스 (오분류 먼저, 정답 나중 — 팀원 방식: 맞은/틀린 섞기)
by_cls = {c: [] for c in CLASSES}
for i, (path, label) in enumerate(zip(test_img_paths, test_labels)):
    by_cls[CLASSES[label]].append((path, all_preds[i]))

model.eval()
for row, cls_name in enumerate(target_cls):
    cls_idx  = CLASSES.index(cls_name)
    items    = by_cls[cls_name]
    # 틀린 것 먼저 최대 2장, 맞은 것으로 채움
    wrong = [(p, pr) for p, pr in items if pr != cls_idx]
    right = [(p, pr) for p, pr in items if pr == cls_idx]
    show  = (wrong[:2] + right)[:PER_CLASS]

    for col, (path, pred_idx) in enumerate(show):
        img_pil  = exif_safe_loader(path)
        img_np   = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE)))
        inp      = eval_tf(img_pil).unsqueeze(0).to(DEVICE)

        cam, _, probs = cam_engine(inp)
        overlay       = overlay_cam(img_np, cam)

        mark  = 'O' if pred_idx == cls_idx else 'X'
        color = 'green' if pred_idx == cls_idx else 'red'
        axes[row][col].imshow(overlay)
        axes[row][col].set_title(
            f'실제: {cls_name}\n예측: {CLASSES[pred_idx]} ({probs[pred_idx]*100:.1f}%) {mark}',
            fontsize=8, color=color)
        axes[row][col].axis('off')

    for col in range(len(show), PER_CLASS):
        axes[row][col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'gradcam_v4.png'), dpi=120, bbox_inches='tight')
plt.show()
print('GradCAM 저장 완료')


## 13. 최종 결과 요약

In [ ]:
from sklearn.metrics import f1_score

macro_f1 = f1_score(test_labels, all_preds, average='macro')
per_f1   = f1_score(test_labels, all_preds, average=None)

print('┌──────────────────────────────────────────────────┐')
print('│          ConvNeXt-Tiny v4 최종 결과              │')
print('├──────────────────────────────────────────────────┤')
print(f'│  Test Accuracy       : {acc*100:>6.2f}%                   │')
print(f'│  Macro F1            : {macro_f1:>6.4f}                   │')
print('├──────────────────────────────────────────────────┤')
for i, cls in enumerate(CLASSES):
    print(f'│  {cls:14s} F1 : {per_f1[i]:>6.4f}                   │')
print('├──────────────────────────────────────────────────┤')
print('│  v3 대비 v4 변경사항                              │')
print('│  ✅ LLRD (레이어별 LR 감쇠 0.8)                   │')
print('│  ✅ RandomErasing (로고·버튼 집중 방지)            │')
print('│  ✅ GradCAM: 팀원 gradcam.py 방식 통일            │')
print('│  ✅ v3 핵심 유지 (RandomPerspective, 224, batch32)│')
print('└──────────────────────────────────────────────────┘')
